# Perth Property Price Prediction - Model Training

**Author:** Haz Li
**Date:** August, 2025

## Objective
The goal of this notebook is to develop a machine learning model that can accurately predict the price of a property in Perth based on its key features. The process involves:
1.  **Loading Data:** Connecting to our structured MySQL database to fetch clean, relational data.
2.  **Feature Engineering:** Preparing the data for modeling, primarily through one-hot encoding of categorical variables.
3.  **Model Training:** Training a `RandomForestRegressor` model, which is well-suited for tabular data and robust to outliers.
4.  **Evaluation:** Assessing the model's performance using the R-squared (R²) metric.
5.  **Serialization:** Saving the trained model and the required data columns for deployment in our Flask web application.

In [2]:
# --- 1. Import Necessary Libraries ---

import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
import joblib  # For saving our model and columns
import warnings
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error

# Ignore warnings for cleaner output
warnings.filterwarnings('ignore')

# Set pandas display options for better viewing
pd.set_option('display.max_columns', 50)

print("Libraries imported successfully.")

Libraries imported successfully.


## 2. Data Loading

We will connect directly to our clean, structured MySQL database. This ensures we are using the "single source of truth" that we established during the ETL phase. The SQL query will join the fact and dimension tables to create a rich, flat dataset suitable for machine learning.

In [6]:
# --- 2. Database Connection and Data Loading (Secure Version) ---

import os
from dotenv import load_dotenv

# Load environment variables from the .env file in the project's root directory.
load_dotenv()
print("Attempting to load environment variables from .env file...")

# Read database credentials securely from environment variables.
DB_USER = os.getenv("DB_USER", "root")
DB_PASS = os.getenv("DB_PASS")
DB_HOST = 'localhost'
DB_PORT = '3306'
DB_NAME = 'perth_property_db'

# A critical check to ensure the password was found.
if not DB_PASS:
    raise ValueError("DB_PASS environment variable not found or is empty. Please ensure your .env file is correctly set up in the project root.")
else:
    print("Database password loaded successfully from environment variable.")

# Create the SQLAlchemy engine using the loaded credentials.
db_connection_str = f'mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
engine = create_engine(db_connection_str)

# SQL query to join all tables and create a comprehensive dataset for modeling.
query = """
    SELECT
        p.price,
        p.land_size,
        p.parking_spaces,
        p.distance_to_cbd,
        p.property_type,
        s.suburb_name,
        l.bedrooms,
        l.bathrooms,
        ps.primary_school_icsea,
        ss.secondary_school_icsea
    FROM
        FACT_Properties p
    JOIN DIM_Suburbs s ON p.suburb_id = s.suburb_id
    JOIN DIM_Layouts l ON p.layout_id = l.layout_id
    LEFT JOIN DIM_Primary_Schools ps ON p.primary_school_id = ps.primary_school_id
    LEFT JOIN DIM_Secondary_Schools ss ON p.secondary_school_id = ss.secondary_school_id;
"""

print("\nLoading data from database...")
# Use a try-except block for robust data loading.
try:
    df = pd.read_sql(text(query), engine)
    print("Data loaded successfully!")
    print(f"Dataset shape: {df.shape}")
except Exception as e:
    print(f"Error loading data: {e}")

# Display the first few rows and info to verify.
display(df.head())
df.info()

Attempting to load environment variables from .env file...
Database password loaded successfully from environment variable.

Loading data from database...
Data loaded successfully!
Dataset shape: (42954, 10)


,price,land_size,parking_spaces,distance_to_cbd,property_type,suburb_name,bedrooms,bathrooms,primary_school_icsea,secondary_school_icsea
0,395000.0,411,1,13800,duplex-semi-detached,Alexander Heights,3,1,996,977
1,625000.0,695,2,14299,house,Alexander Heights,6,3,994,1030
2,275000.0,181,1,13940,villa,Alexander Heights,3,1,994,1030
3,455000.0,547,2,14609,house,Alexander Heights,4,2,994,1030
4,450000.0,714,3,13816,house,Alexander Heights,4,2,994,1010


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42954 entries, 0 to 42953
Data columns (total 10 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   price                   42954 non-null  float64
 1   land_size               42954 non-null  int64  
 2   parking_spaces          42954 non-null  int64  
 3   distance_to_cbd         42954 non-null  int64  
 4   property_type           42954 non-null  object 
 5   suburb_name             42954 non-null  object 
 6   bedrooms                42954 non-null  int64  
 7   bathrooms               42954 non-null  int64  
 8   primary_school_icsea    42954 non-null  int64  
 9   secondary_school_icsea  42954 non-null  int64  
dtypes: float64(1), int64(7), object(2)
memory usage: 3.3+ MB


## 3. Feature Engineering v3: Value-Based Segmentation via Residual Analysis

After critical review, it was determined that segmenting suburbs based on raw median price is inherently flawed. A high median price could be caused by larger properties, not necessarily higher intrinsic location value. To solve this, we are implementing a sophisticated, two-stage modeling approach to engineer a "Pure Location Value" feature.

**The Process: A Two-Stage Residual Analysis**

1.  **Stage 1: Train a "Location-Agnostic" Base Model.**
    *   First, we train a RandomForest model to predict property prices based *only* on their physical attributes (bedrooms, bathrooms, land size, property type, etc.), deliberately excluding any direct location features like suburb name or postcode.
    *   This model learns the "average" price for a property of a certain type and size, across the entire Perth market.

2.  **Stage 2: Calculate the "Location Premium" (Residuals).**
    *   We use this trained base model to predict the price of every property in our dataset.
    *   The **residual** (i.e., `actual_price - predicted_price`) is then calculated for each property. This residual represents the price difference attributable to factors the model *didn't* see—primarily, the value of its location.
    *   We then calculate the **average residual for each suburb**. This "average location premium" is a far more accurate and fair measure of a suburb's intrinsic market value than raw median price.

3.  **Final Step: Create Tiers from the Location Premium.**
    *   These calculated location premiums are then used to segment suburbs into our final price tiers for the main predictive model.

In [11]:
# --- 3. Feature Engineering and Preparation  ---

# --- 3a. Prepare Data for the Base Model (to calculate location premium) ---
# Select ONLY physical attributes, excluding any direct location identifiers.
base_model_features = [
    'bedrooms', 'bathrooms', 'land_size', 'parking_spaces', 'distance_to_cbd',
    'primary_school_icsea', 'secondary_school_icsea', 'property_type'
]
# We still need 'suburb_name' and 'price' for grouping and calculating residuals.
df_base = df[['suburb_name', 'price'] + base_model_features].dropna()

# One-hot encode 'property_type' for the base model.
df_base_encoded = pd.get_dummies(df_base, columns=['property_type'], prefix='type', drop_first=True)

X_base = df_base_encoded.drop(['price', 'suburb_name'], axis=1)
y_base = df_base_encoded['price']

# --- 3b. Train the "Location-Agnostic" Base Model ---
print("--- Stage 1: Training Location-Agnostic Base Model ---")
base_model = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1, max_depth=15)
base_model.fit(X_base, y_base)
print("Base model training complete.")

# --- 3c. Calculate Residuals (Location Premium) ---
print("\n--- Stage 2: Calculating Location Premiums (Residuals) ---")
all_predictions = base_model.predict(X_base)
df_base_encoded['location_premium'] = df_base_encoded['price'] - all_predictions
suburb_premiums = df_base_encoded.groupby('suburb_name')['location_premium'].mean()
print("Average location premium calculated for each suburb.")


# --- 3d. Create Final Tiers AND the Mapping Dictionary ---
# Use qcut to segment suburbs into 4 tiers based on their calculated premium.
price_tiers = pd.qcut(suburb_premiums, 4, labels=['Standard Value', 'Good Value', 'High Value', 'Premium Value'])

# This is the critical map we need for our Flask app.
price_tier_map = price_tiers.to_dict()

# Save the price_tier_map for the Flask app.
joblib.dump(price_tier_map, 'price_tier_map.pkl')
print("\nPrice tier map saved successfully to price_tier_map.pkl!")
display(price_tiers.head())
# ======================================================================

# --- 3e. Map the new tier back to our original DataFrame ---
# We use the original 'df' here to ensure we have all the rows.
df['suburb_value_tier'] = df['suburb_name'].map(price_tier_map)
print("\nFinal market tiers created and mapped back to the main DataFrame.")

# --- 3f. Final Feature Preparation for the Main Model ---
final_features_to_use = [
    'bedrooms', 'bathrooms', 'land_size', 'parking_spaces', 'distance_to_cbd',
    'primary_school_icsea', 'secondary_school_icsea',
    'suburb_value_tier' # Use our new, ultra-robust feature
]
target = 'price'

df_model = df[final_features_to_use + [target]].dropna()
df_model_encoded = pd.get_dummies(df_model, columns=['suburb_value_tier'], prefix='tier', drop_first=True)

print("\nFinal categorical tier feature has been one-hot encoded.")
X = df_model_encoded.drop(target, axis=1)
y = df_model_encoded[target]

print(f"\nShape of final X (features): {X.shape}")
print(f"Shape of final y (target): {y.shape}")

--- Stage 1: Training Location-Agnostic Base Model ---
Base model training complete.

--- Stage 2: Calculating Location Premiums (Residuals) ---
Average location premium calculated for each suburb.

Price tier map saved successfully to price_tier_map.pkl!


suburb_name
Alexander Heights    Standard Value
Alfred Cove              Good Value
Applecross            Premium Value
Ardross              Standard Value
Ascot                    High Value
Name: location_premium, dtype: category
Categories (4, object): ['Standard Value' < 'Good Value' < 'High Value' < 'Premium Value']


Final market tiers created and mapped back to the main DataFrame.

Final categorical tier feature has been one-hot encoded.

Shape of final X (features): (42954, 10)
Shape of final y (target): (42954,)


In [12]:
# --- 4. Final Model Training & Evaluation ---

# --- 4a. Train/Test Split ---
# We split our final, engineered dataset into 80% for training and 20% for testing.
# This ensures we evaluate the model on data it has never seen before.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Testing set size: {X_test.shape[0]} samples")

# --- 4b. Initialize and Train the Final Model ---
# We can use more estimators (e.g., 200) for our final model for potentially higher accuracy.
# n_jobs=-1 uses all available CPU cores for faster training.
print("\n--- Training the Final RandomForestRegressor Model ---")
final_model = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1, oob_score=True)
final_model.fit(X_train, y_train)
print("Final model training complete.")

# --- 4c. Evaluate the Final Model's Performance ---
# The OOB (Out-of-Bag) score is a great internal estimate of the model's performance.
print(f"\nModel Out-of-Bag (OOB) Score: {final_model.oob_score_:.4f}")

# Make predictions on the unseen test set.
print("\nEvaluating model on the test set...")
y_pred = final_model.predict(X_test)

# Calculate key performance metrics.
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"Test Set R-squared (R²) Score: {r2:.4f}")
print(f"Test Set Root Mean Squared Error (RMSE): ${rmse:,.2f}")

# --- 4d. Feature Importance (Insight!) ---
# Let's see what the model thinks are the most important features.
print("\n--- Top 10 Most Important Features ---")
feature_importances = pd.Series(final_model.feature_importances_, index=X.columns).sort_values(ascending=False)
display(feature_importances.head(10))

Training set size: 34363 samples
Testing set size: 8591 samples

--- Training the Final RandomForestRegressor Model ---
Final model training complete.

Model Out-of-Bag (OOB) Score: 0.7894

Evaluating model on the test set...
Test Set R-squared (R²) Score: 0.7972
Test Set Root Mean Squared Error (RMSE): $260,464.07

--- Top 10 Most Important Features ---


primary_school_icsea      0.306892
bathrooms                 0.222181
land_size                 0.189033
distance_to_cbd           0.098274
tier_Premium Value        0.063198
secondary_school_icsea    0.054348
bedrooms                  0.033594
parking_spaces            0.023320
tier_High Value           0.006112
tier_Standard Value       0.003047
dtype: float64

In [13]:
# --- 5. Save the Model and Columns for Deployment ---

# Get the final list of columns from our feature matrix X.
model_columns = X.columns

# Define the file paths.
model_path = 'property_price_predictor.pkl'
columns_path = 'model_columns.pkl'

# Save the trained model using joblib for efficiency.
joblib.dump(final_model, model_path)
print(f"Final trained model saved to: {model_path}")

# Save the list of feature columns. This is CRITICAL for the web app.
joblib.dump(model_columns, columns_path)
print(f"Model columns saved to: {columns_path}")

print("\nNotebook execution complete. The final model and columns are ready for deployment in the Flask app.")



Final trained model saved to: property_price_predictor.pkl
Model columns saved to: model_columns.pkl

Notebook execution complete. The final model and columns are ready for deployment in the Flask app.


In [14]:
joblib.dump(price_tier_map, 'price_tier_map.pkl')
print("Price tier map saved successfully to price_tier_map.pkl!")

Price tier map saved successfully to price_tier_map.pkl!
